In [0]:
%run ../config_setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

# 1. CALENDAR DIMENSION (Date-based KPIs ke liye base)
date_df = spark.sql("""
    SELECT explode(sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day)) as full_date
""").select(
    date_format(col("full_date"), "yyyyMMdd").cast("int").alias("date_sk"),
    col("full_date"),
    year("full_date").alias("year"),
    quarter("full_date").alias("quarter"),
    month("full_date").alias("month"),
    weekofyear("full_date").alias("week_of_year"),
    dayofweek("full_date").alias("day_of_week"),
    date_format("full_date", "EEEE").alias("day_name"),
    when(dayofweek("full_date").isin([1, 7]), True).otherwise(False).alias("is_weekend")
)
date_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_date")

# 2. TIME BUCKET (30-min Tumbling Window base)
time_df = spark.sql("""
    SELECT explode(sequence(to_timestamp('2020-01-01 00:00:00'), to_timestamp('2020-01-01 23:30:00'), interval 30 minutes)) as time_start
""").select(
    date_format(col("time_start"), "HHmm").cast("int").alias("time_bucket_sk"),
    col("time_start"),
    (col("time_start") + expr("INTERVAL 30 MINUTES")).alias("time_end")
)

time_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD}.dim_time_30min")

print("✅ gold_01_dim_static built successfully!")